# 📊 Exploratory Data Analysis (EDA)
## GoEmotions-Ekman: Multi-label Emotion Classification

### Tujuan EDA
EDA ini berorientasi pada **keputusan ML**, bukan sekadar deskriptif.

| Analisis | Tujuan ML |
|----------|----------|
| Frekuensi & proporsi label | Mengidentifikasi label dominan dan langka |
| Panjang teks | Menentukan max_sequence_length dan potensi truncation |
| Label cardinality | Mengukur rata-rata jumlah label aktif per teks |
| Label co-occurrence | Memahami label yang sering muncul bersama |
| Contoh kasus sulit | Mendeteksi ambiguitas sebelum modeling |

In [ ]:
# Setup
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from src.data_loader import load_goemotions, validate_dataset, get_splits_as_dataframe, print_validation_report
from src.config import LABEL_COLUMNS, TEXT_COLUMN

# Style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('Setup complete!')
print(f'Labels: {LABEL_COLUMNS}')

## 1. Load & Validate Dataset

In [ ]:
# Load dataset
ds = load_goemotions()
print(ds)

In [ ]:
# Data Validation
validation_results = validate_dataset(ds)
print_validation_report(validation_results)

In [ ]:
# Convert to DataFrames
dfs = get_splits_as_dataframe(ds)

df_train = dfs['train']
df_val = dfs['validation']
df_test = dfs['test']

print(f'Train: {len(df_train)} samples')
print(f'Validation: {len(df_val)} samples')
print(f'Test: {len(df_test)} samples')
print(f'\nColumns: {list(df_train.columns)}')
df_train.head()

## 2. Frekuensi & Proporsi 7 Label Emosi
**Tujuan ML:** Mengidentifikasi label dominan dan label langka yang mungkin sulit diprediksi.

In [ ]:
# Hitung frekuensi per label
label_counts = df_train[LABEL_COLUMNS].sum().sort_values(ascending=False)
label_proportions = label_counts / len(df_train)

print('Label Frequencies (Train):')
for label, count in label_counts.items():
    print(f'  {label:<12}: {count:>6} ({label_proportions[label]:.2%})')

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
label_counts.plot(kind='bar', ax=ax1, color=sns.color_palette('Set2', 7))
ax1.set_title('Label Frequency (Train Set)')
ax1.set_ylabel('Count')
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha='right')
for i, v in enumerate(label_counts.values):
    ax1.text(i, v + 50, str(v), ha='center', fontsize=10)

ax2 = axes[1]
label_proportions.plot(kind='bar', ax=ax2, color=sns.color_palette('Set2', 7))
ax2.set_title('Label Proportion (Train Set)')
ax2.set_ylabel('Proportion')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45, ha='right')
for i, v in enumerate(label_proportions.values):
    ax2.text(i, v + 0.005, f'{v:.2%}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('../outputs/eda_label_frequency.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Single-label vs Multi-label Analysis
**Tujuan ML:** Memahami proporsi teks yang memiliki lebih dari satu emosi aktif.

In [ ]:
# Hitung jumlah label aktif per teks
df_train['num_labels'] = df_train[LABEL_COLUMNS].sum(axis=1)

label_count_dist = df_train['num_labels'].value_counts().sort_index()
print('Distribution of number of active labels per text:')
for n, count in label_count_dist.items():
    print(f'  {int(n)} label(s): {count:>6} ({count/len(df_train):.2%})')

single_label = (df_train['num_labels'] == 1).sum()
multi_label = (df_train['num_labels'] > 1).sum()
no_label = (df_train['num_labels'] == 0).sum()

print(f'\nNo label:     {no_label:>6} ({no_label/len(df_train):.2%})')
print(f'Single-label: {single_label:>6} ({single_label/len(df_train):.2%})')
print(f'Multi-label:  {multi_label:>6} ({multi_label/len(df_train):.2%})')

# Bar chart
fig, ax = plt.subplots(figsize=(8, 5))
label_count_dist.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Distribution of Active Labels per Text')
ax.set_xlabel('Number of Active Labels')
ax.set_ylabel('Count')
for i, (n, count) in enumerate(label_count_dist.items()):
    ax.text(i, count + 50, str(count), ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('../outputs/eda_label_count_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Label Cardinality
**Tujuan ML:** Rata-rata jumlah label aktif per teks. Memberikan gambaran tentang seberapa "multi" multi-label dataset ini.

In [ ]:
label_cardinality_train = df_train[LABEL_COLUMNS].sum(axis=1).mean()
label_cardinality_val = df_val[LABEL_COLUMNS].sum(axis=1).mean()
label_cardinality_test = df_test[LABEL_COLUMNS].sum(axis=1).mean()

print(f'Label Cardinality:')
print(f'  Train:      {label_cardinality_train:.4f}')
print(f'  Validation: {label_cardinality_val:.4f}')
print(f'  Test:       {label_cardinality_test:.4f}')
print(f'\nInterpretasi: Rata-rata setiap teks memiliki ~{label_cardinality_train:.2f} label aktif.')

## 5. Label Co-occurrence Matrix / Heatmap
**Tujuan ML:** Memahami pasangan emosi yang sering muncul bersama. Ini penting untuk memahami error patterns.

In [ ]:
# Co-occurrence matrix
label_matrix = df_train[LABEL_COLUMNS].values
co_occurrence = label_matrix.T @ label_matrix
co_occurrence_df = pd.DataFrame(co_occurrence, index=LABEL_COLUMNS, columns=LABEL_COLUMNS)

print('Label Co-occurrence Matrix:')
print(co_occurrence_df)

# Heatmap
fig, ax = plt.subplots(figsize=(8, 7))
# Mask diagonal for better visualization of co-occurrences
mask = np.eye(len(LABEL_COLUMNS), dtype=bool)
sns.heatmap(
    co_occurrence_df, 
    annot=True, 
    fmt='d', 
    cmap='YlOrRd',
    mask=mask,
    ax=ax,
    linewidths=0.5,
)
ax.set_title('Label Co-occurrence Heatmap (off-diagonal)')
plt.tight_layout()
plt.savefig('../outputs/eda_co_occurrence.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Panjang Komentar & Implikasi Max Sequence Length
**Tujuan ML:** Menentukan `max_sequence_length` yang optimal — trade-off antara informasi yang dipertahankan dan biaya komputasi.

In [ ]:
# Panjang dalam karakter dan kata
df_train['text_len_chars'] = df_train[TEXT_COLUMN].str.len()
df_train['text_len_words'] = df_train[TEXT_COLUMN].str.split().str.len()

print('Text Length Statistics (characters):')
print(df_train['text_len_chars'].describe())
print(f'\nText Length Statistics (words):')
print(df_train['text_len_words'].describe())

# Percentiles yang relevan untuk max_length decision
for p in [90, 95, 98, 99, 99.5, 100]:
    val = df_train['text_len_words'].quantile(p/100)
    print(f'  P{p}: {val:.0f} words')

# Distribution plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_train['text_len_words'], bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(x=df_train['text_len_words'].quantile(0.95), color='red', linestyle='--', label='P95')
axes[0].set_title('Word Count Distribution')
axes[0].set_xlabel('Number of Words')
axes[0].set_ylabel('Count')
axes[0].legend()

axes[1].hist(df_train['text_len_chars'], bins=50, color='darkorange', edgecolor='white')
axes[1].axvline(x=df_train['text_len_chars'].quantile(0.95), color='red', linestyle='--', label='P95')
axes[1].set_title('Character Count Distribution')
axes[1].set_xlabel('Number of Characters')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/eda_text_length.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Tokenizer-based length analysis (more accurate for Transformer)
from src.preprocessing import get_tokenizer

tokenizer = get_tokenizer()

# Tokenize sample to check actual token lengths
token_lengths = []
for text in df_train[TEXT_COLUMN].values:
    tokens = tokenizer(text, add_special_tokens=True)
    token_lengths.append(len(tokens['input_ids']))

df_train['token_length'] = token_lengths

print('Token Length Statistics (BERT tokenizer):')
print(pd.Series(token_lengths).describe())

for p in [90, 95, 98, 99, 99.5, 100]:
    val = np.percentile(token_lengths, p)
    print(f'  P{p}: {val:.0f} tokens')

# Candidate max_sequence_length analysis
for max_len in [64, 96, 128, 192, 256]:
    truncated = sum(1 for l in token_lengths if l > max_len)
    print(f'  max_length={max_len}: {truncated} texts truncated ({truncated/len(token_lengths):.2%})')

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(token_lengths, bins=50, color='teal', edgecolor='white')
for max_len, color in [(64, 'orange'), (128, 'red'), (256, 'purple')]:
    ax.axvline(x=max_len, color=color, linestyle='--', label=f'max_length={max_len}')
ax.set_title('BERT Token Length Distribution')
ax.set_xlabel('Number of Tokens')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/eda_token_length.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Identifikasi Label Langka & Pasangan Emosi yang Sering Co-occur
**Tujuan ML:** Label langka kemungkinan besar memiliki recall rendah. Pasangan yang sering co-occur bisa mempengaruhi error patterns.

In [ ]:
# Label langka (sorted by frequency)
print('Labels sorted by frequency (ascending = paling langka):')
label_freq = df_train[LABEL_COLUMNS].sum().sort_values()
for label, count in label_freq.items():
    imbalance_ratio = len(df_train) / count if count > 0 else float('inf')
    print(f'  {label:<12}: {count:>6} (imbalance ratio: 1:{imbalance_ratio:.1f})')

# Top co-occurring pairs
print('\nTop 10 co-occurring label pairs:')
pair_counts = []
for i in range(len(LABEL_COLUMNS)):
    for j in range(i+1, len(LABEL_COLUMNS)):
        count = ((df_train[LABEL_COLUMNS[i]] == 1) & (df_train[LABEL_COLUMNS[j]] == 1)).sum()
        if count > 0:
            pair_counts.append((LABEL_COLUMNS[i], LABEL_COLUMNS[j], count))

pair_counts.sort(key=lambda x: x[2], reverse=True)
for l1, l2, count in pair_counts[:10]:
    print(f'  {l1} + {l2}: {count}')

## 8. Contoh Kasus Sulit / Ambigu
**Tujuan ML:** Mendeteksi ambiguitas, emosi implisit, ironi, atau noise sebelum modeling.

In [ ]:
# Contoh multi-label (lebih dari 1 emosi)
multi_label_examples = df_train[df_train['num_labels'] > 1].head(10)
print('=== Multi-label Examples ===')
for _, row in multi_label_examples.iterrows():
    active = [l for l in LABEL_COLUMNS if row[l] == 1]
    print(f'  Text: "{row[TEXT_COLUMN][:100]}..."')
    print(f'  Labels: {active}')
    print()

In [ ]:
# Contoh setiap emosi
print('=== Sample per Emotion ===')
for label in LABEL_COLUMNS:
    sample = df_train[df_train[label] == 1].head(2)
    print(f'\n--- {label.upper()} ---')
    for _, row in sample.iterrows():
        active = [l for l in LABEL_COLUMNS if row[l] == 1]
        print(f'  "{row[TEXT_COLUMN][:120]}"')
        print(f'  All labels: {active}')

## 9. Distribusi Label per Split (Consistency Check)
**Tujuan ML:** Memastikan distribusi label konsisten antar split agar tidak ada bias.

In [ ]:
# Proporsi label per split
split_props = pd.DataFrame({
    'Train': df_train[LABEL_COLUMNS].mean(),
    'Validation': df_val[LABEL_COLUMNS].mean(),
    'Test': df_test[LABEL_COLUMNS].mean(),
})
print('Label Proportions per Split:')
print(split_props.round(4))

# Grouped bar chart
fig, ax = plt.subplots(figsize=(12, 5))
split_props.plot(kind='bar', ax=ax)
ax.set_title('Label Proportions Across Splits')
ax.set_ylabel('Proportion')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.legend(title='Split')
plt.tight_layout()
plt.savefig('../outputs/eda_split_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Ringkasan EDA & Keputusan ML

| Aspek | Temuan | Keputusan ML |
|-------|--------|-------------|
| Label dominan | *(isi setelah analisis)* | Perhatikan Micro vs Macro F1 |
| Label langka | *(isi setelah analisis)* | Ekspektasi recall rendah pada label langka |
| Label cardinality | *(isi setelah analisis)* | Justifikasi multi-label formulation |
| Co-occurrence | *(isi setelah analisis)* | Pengaruh pada error analysis |
| Max sequence length | *(isi setelah analisis)* | Pilih berdasarkan P95/P98 token length |
| Split balance | *(isi setelah analisis)* | Konfirmasi split dapat diandalkan |